In [ ]:
import cv2
import numpy as np
from keras.models import load_model
from PIL import Image
import pickle
import matplotlib.pyplot as plt
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL
import io

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


# Load the trained model
model = load_model('/content/drive/MyDrive/face_recognition_model_transfer_learning.keras')
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
pickle_in = open("/content/drive/MyDrive/X_train.pickle","rb")
X = pickle.load(pickle_in)

pickle_in = open("/content/drive/MyDrive/Y_train.pickle","rb")
y = pickle.load(pickle_in)

In [ ]:
# Load the Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

#modified
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Define label names mapping for camera section
label_names = {
    0: 'Akram',
    1: 'Bisal',
    2: 'Tawhid'
    # Add more as needed for additional labels
}
# Load the training data (X and y) and class labels
pickle_in = open("/content/drive/MyDrive/X_train.pickle", "rb")
X = pickle.load(pickle_in)  # Images

pickle_in = open("/content/drive/MyDrive/Y_train.pickle", "rb")
y = pickle.load(pickle_in)  # Labels

# Load class labels (names)
with open('/content/drive/MyDrive/class_labels.pickle', 'rb') as f:
    class_labels = pickle.load(f)

print("Loaded Class Labels:", class_labels)
print("Loaded Labels:", class_labels)
# m end
if isinstance(y, int):
    y = [y]
elif not isinstance(y, (list, np.ndarray)):
    raise ValueError("Unexpected format for labels in 'y'. Expected list or array-like.")

if isinstance(y, np.ndarray):
    y = y.tolist()

class_labels = sorted(set(y))  # Ensure unique labels and sorted order
print("Loaded Labels:", class_labels)

# Inspect the input shape of the model
print("Model input shape:", model.input_shape)

# Function to preprocess the face for the model
def preprocess_face(face, target_shape):
    face_resized = cv2.resize(face, (target_shape[1], target_shape[2]))
    face_normalized = face_resized / 255.0
    face_array = np.expand_dims(face_normalized, axis=0)  # Add batch dimension
    if len(model.input_shape) == 2:  # If the model expects flattened input
        face_array = face_array.reshape((1, -1))  # Flatten the array
    return face_array

# Function to handle predictions
def predict_label(face_array):
    predictions = model.predict(face_array)
    confidence = np.max(predictions)  # Get the maximum confidence
    label_index = np.argmax(predictions)  # Get the index of the highest confidence class

    # Print only confidence and label
    print(f"Label: {class_labels[label_index]}, Confidence: {confidence*100:.2f}%")

    return label_index, confidence

# Function to convert JavaScript object to OpenCV image
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

# Function to convert OpenCV bounding box to base64 for overlay
def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes

#  for video stream
def video_stream():
    js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;

        var pendingResolve = null;
        var shutdown = false;

        function removeDom() {
            stream.getVideoTracks()[0].stop();
            video.remove();
            div.remove();
            video = null;
            div = null;
            stream = null;
            imgElement = null;
            captureCanvas = null;
            labelElement = null;
        }

        function onAnimationFrame() {
            if (!shutdown) {
                window.requestAnimationFrame(onAnimationFrame);
            }
            if (pendingResolve) {
                var result = "";
                if (!shutdown) {
                    captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
                    result = captureCanvas.toDataURL('image/jpeg', 0.8);
                }
                var lp = pendingResolve;
                pendingResolve = null;
                lp(result);
            }
        }

        async function createDom() {
            if (div !== null) {
                return stream;
            }

            div = document.createElement('div');
            div.style.border = '2px solid black';
            div.style.padding = '3px';
            div.style.width = '100%';
            div.style.maxWidth = '600px';
            document.body.appendChild(div);

            const modelOut = document.createElement('div');
            modelOut.innerHTML = "Status:";
            labelElement = document.createElement('span');
            labelElement.innerText = 'No data';
            labelElement.style.fontWeight = 'bold';
            modelOut.appendChild(labelElement);
            div.appendChild(modelOut);

            video = document.createElement('video');
            video.style.display = 'block';
            video.width = div.clientWidth - 6;
            video.setAttribute('playsinline', '');
            video.onclick = () => { shutdown = true; };
            stream = await navigator.mediaDevices.getUserMedia(
                {video: { facingMode: "environment"}});
            div.appendChild(video);

            imgElement = document.createElement('img');
            imgElement.style.position = 'absolute';
            imgElement.style.zIndex = 1;
            imgElement.onclick = () => { shutdown = true; };
            div.appendChild(imgElement);

            const instruction = document.createElement('div');
            instruction.innerHTML =
                '' +
                'When finished, click here or on the video to stop this demo';
            div.appendChild(instruction);
            instruction.onclick = () => { shutdown = true; };

            video.srcObject = stream;
            await video.play();

            captureCanvas = document.createElement('canvas');
            captureCanvas.width = 640;
            captureCanvas.height = 480;
            window.requestAnimationFrame(onAnimationFrame);

            return stream;
        }

        async function stream_frame(label, imgData) {
            if (shutdown) {
                removeDom();
                shutdown = false;
                return '';
            }

            var preCreate = Date.now();
            stream = await createDom();

            var preShow = Date.now();
            if (label != "") {
                labelElement.innerHTML = label;
            }

            if (imgData != "") {
                var videoRect = video.getClientRects()[0];
                imgElement.style.top = videoRect.top + "px";
                imgElement.style.left = videoRect.left + "px";
                imgElement.style.width = videoRect.width + "px";
                imgElement.style.height = videoRect.height + "px";
                imgElement.src = imgData;
            }

            var preCapture = Date.now();
            var result = await new Promise(function(resolve, reject) {
                pendingResolve = resolve;
            });
            shutdown = false;

            return {'create': preShow - preCreate,
                    'show': preCapture - preShow,
                    'capture': Date.now() - preCapture,
                    'img': result};
        }
    ''')
    display(js)

def video_frame(label, bbox):
    data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
    return data

# Start video stream
video_stream()
label_html = 'Capturing...'
bbox = ''
count = 0

while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break

    # Convert JS response to OpenCV image
    img = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480, 640, 4], dtype=np.uint8)

    # Convert image to grayscale for face detection
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    for (x, y, w, h) in faces:
        # Extract and preprocess the detected face
        face = img[y:y+h, x:x+w]
        face_array = preprocess_face(face, target_shape=model.input_shape)

        # Predict the class and confidence
        label_index, confidence = predict_label(face_array)

        # Determine label
        if confidence > 0.5:  # Confidence threshold
           # label = f"{class_labels[label_index]} ({confidence*100:.2f}%)"
            label = f"{label_names[label_index]} ({confidence*100:.2f}%)"
            color = (0, 255, 0)  # Green for recognized


        else:
            label = "Whoooo??"
            color = (0, 0, 0)  # Red for unknown



        # Draw bounding box and label
        bbox_array = cv2.rectangle(bbox_array, (x, y), (x+w, y+h), color, 2)
        bbox_array = cv2.putText(bbox_array, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)


# Convert `img` to base64 format to send it back to JavaScript
    _, buffer = cv2.imencode('.jpg', img)
    bbox_bytes = 'data:image/jpeg;base64,' + b64encode(buffer).decode()

    # Convert bbox to overlay
    bbox_array[:, :, 3] = (bbox_array.max(axis=2) > 0).astype(int) * 255
    bbox_bytes = bbox_to_bytes(bbox_array)
    bbox = bbox_bytes

Loaded Class Labels: ['Tawhid', 'Akram', 'Bishal']
Loaded Labels: ['Tawhid', 'Akram', 'Bishal']
Loaded Labels: [0, 1, 2]
Model input shape: (None, 224, 224, 3)


<IPython.core.display.Javascript object>

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Label: 2, Confidence: 78.44%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Label: 2, Confidence: 80.42%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Label: 2, Confidence: 69.22%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Label: 2, Confidence: 76.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Label: 2, Confidence: 44.15%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Label: 2, Confidence: 69.88%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Label: 2, Confidence: 48.06%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Label: 2, Confidence: 69.29%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Label: 1, Confidence: 35.01%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Label: 2, Confidence: 64.93%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Label: 2, Confidence: 39.98%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Label: 2, Confidence: 84.66%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Label: 2, Confidence: 49.58%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Label: 2, Confidence: 66.83%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Label: 0, Confidence: 50

KeyboardInterrupt: 

#**Camera** **ON**

Test tawhid

doneeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee